In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Change this if you put the shortcut somewhere else
BASE = "/content/drive/MyDrive/255-GroupProject"

TRAIN_JSONL = f"{BASE}/FindVehicle_train.jsonl"   # your train.jsonl
OUT_CSV      = f"{BASE}/FindVehicle_train.csv"     # output CSV you want


In [ ]:
import json, csv
from pathlib import Path

def jsonl_to_token_csv(in_path, out_path):
    """
    Writes a CSV with columns: Description, token, tag.
    tag = "0" for non-entity tokens; entity spans become B/I/E-<type>.
    Assumes:
      ex["data"] is the sentence
      ex["ner_label"] entries look like:
        [etype, char_start, char_end, surface, tok_start, tok_end, variants]
      where token span is [tok_start, tok_end) in token indices.
    Tokenization: text.split() (your data has spaced punctuation, so this matches the TXT).
    """
    in_path = Path(in_path)
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with in_path.open("r", encoding="utf-8") as fin, \
         out_path.open("w", newline="", encoding="utf-8-sig") as fout:
        writer = csv.DictWriter(fout, fieldnames=["Description", "token", "tag"])
        writer.writeheader()

        for line in fin:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)

            text = ex["data"]
            tokens = text.split()
            tags = ["0"] * len(tokens)  # your requested default tag

            for lbl in ex.get("ner_label", []):
                # Expected list format; includes token start/end
                # [etype, char_start, char_end, surface, tok_start, tok_end, variants]
                if isinstance(lbl, list) and len(lbl) >= 6:
                    etype = lbl[0]
                    ts = lbl[4]
                    te = lbl[5]
                elif isinstance(lbl, dict):
                    etype = lbl.get("type") or lbl.get("etype")
                    ts = lbl.get("tok_start") or lbl.get("ts")
                    te = lbl.get("tok_end") or lbl.get("te")
                else:
                    continue

                if etype is None or ts is None or te is None:
                    continue

                span_len = te - ts
                if span_len <= 0:
                    continue

                if span_len == 1:
                    if 0 <= ts < len(tags):
                        tags[ts] = f"B-{etype}"
                else:
                    if 0 <= ts < len(tags):
                        tags[ts] = f"B-{etype}"
                    for i in range(ts + 1, te - 1):
                        if 0 <= i < len(tags):
                            tags[i] = f"I-{etype}"
                    if 0 <= te - 1 < len(tags):
                        tags[te - 1] = f"E-{etype}"

            # one row per token
            for tok, tag in zip(tokens, tags):
                writer.writerow({
                    "Description": text,
                    "token": tok,
                    "tag": tag
                })

# Run it for your train file
jsonl_to_token_csv(TRAIN_JSONL, OUT_CSV)
print("Wrote:", OUT_CSV)


Wrote: /content/drive/MyDrive/255-GroupProject/FindVehicle_train.csv


In [ ]:
# --- 1) Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

# --- 2) Paths (edit if your shortcut path is different) ---
BASE = "/content/drive/MyDrive/255-GroupProject"
IN_TEST_JSONL = f"{BASE}/FindVehicle_test.jsonl"
OUT_TEST_CSV  = f"{BASE}/FindVehicle_test.csv"

# --- 3) Converter (reuse for any split) ---
import json, csv
from pathlib import Path

def jsonl_to_token_csv(in_path, out_path):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    # Check if the input file exists
    if not Path(in_path).exists():
        print(f"Error: Input file not found at {in_path}")
        return # Exit the function if the file doesn't exist

    with open(in_path, "r", encoding="utf-8") as fin, \
         open(out_path, "w", newline="", encoding="utf-8-sig") as fout:
        writer = csv.DictWriter(fout, fieldnames=["Description", "token", "tag"])
        writer.writeheader()

        for line in fin:
            line = line.strip()
            if not line:
                continue
            ex = json.loads(line)
            text = ex["data"]
            tokens = text.split()              # punctuation already spaced
            tags = ["0"] * len(tokens)         # default tag = "0"

            # Apply NER spans if present
            for lbl in ex.get("ner_label", []):
                # Expected list: [etype, char_start, char_end, surface, tok_start, tok_end, ...]
                if isinstance(lbl, list) and len(lbl) >= 6:
                    etype, ts, te = lbl[0], lbl[4], lbl[5]
                elif isinstance(lbl, dict):     # fallback for dict labels
                    etype = lbl.get("type") or lbl.get("etype")
                    ts, te = lbl.get("tok_start"), lbl.get("tok_end")
                else:
                    continue
                if etype is None or ts is None or te is None or te <= ts:
                    continue

                # BIOE tagging
                tags[ts] = f"B-{etype}"
                for i in range(ts + 1, te - 1):
                    if 0 <= i < len(tags):
                        tags[i] = f"I-{etype}"
                last = te - 1
                if 0 <= last < len(tags):
                    tags[last] = f"E-{etype}"

            for tok, tag in zip(tokens, tags):
                writer.writerow({"Description": text, "token": tok, "tag": tag})

# --- 4) Run for TEST ---
jsonl_to_token_csv(IN_TEST_JSONL, OUT_TEST_CSV)
print(" Wrote:", OUT_TEST_CSV)

# (Optional) peek
import pandas as pd
# Check if the output file exists before attempting to read it
if Path(OUT_TEST_CSV).exists():
    display(pd.read_csv(OUT_TEST_CSV).head(25))
else:
    print(f"Output file not created: {OUT_TEST_CSV}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Wrote: /content/drive/MyDrive/255-GroupProject/FindVehicle_test.csv


,Description,token,tag
0,Let the wise man assist me to find out the Sil...,Let,0
1,Let the wise man assist me to find out the Sil...,the,0
2,Let the wise man assist me to find out the Sil...,wise,0
3,Let the wise man assist me to find out the Sil...,man,0
4,Let the wise man assist me to find out the Sil...,assist,0
5,Let the wise man assist me to find out the Sil...,me,0
6,Let the wise man assist me to find out the Sil...,to,0
7,Let the wise man assist me to find out the Sil...,find,0
8,Let the wise man assist me to find out the Sil...,out,0
9,Let the wise man assist me to find out the Sil...,the,0


In [ ]:
!head -n 30 /content/fv_prepared_cleaned/train.conll


Please	0
help	0
me	0
find	0
the	0
black	B-vehicle_color
Honda	B-vehicle_brand
T360	B-vehicle_model
on	0
the	0
Top-Left	B-vehicle_location
of	0
the	0
image	0
and	0
the	0
peach	B-vehicle_color
blossom	E-vehicle_color
Ford	B-vehicle_brand
Ford	B-vehicle_model
Endura	E-vehicle_model
on	0
the	0
bottom	B-vehicle_location
right	E-vehicle_location
.	0

Let	0
the	0
clever	0


**Convert CSV → CoNLL format (for Flair)**

In [ ]:
import pandas as pd, os

train_csv = "/content/drive/MyDrive/255-GroupProject/FindVehicle_train.csv"
test_csv  = "/content/drive/MyDrive/255-GroupProject/FindVehicle_test.csv"
out_dir = "/content/fv_prepared_cleaned"
os.makedirs(out_dir, exist_ok=True)

def csv_to_conll_clean(csv_path, out_path):
    df = pd.read_csv(csv_path)
    df["token"] = df["token"].astype(str).str.strip()
    df["tag"] = df["tag"].astype(str).replace({"0": "O"}).str.strip()  # 🔹 FIX HERE: convert 0 → O
    with open(out_path, "w", encoding="utf-8") as f:
        current_desc = df.iloc[0]["Description"]
        for _, row in df.iterrows():
            if row["Description"] != current_desc:
                f.write("\n")
                current_desc = row["Description"]
            f.write(f"{row['token']}\t{row['tag']}\n")
        f.write("\n")
    print(f"Wrote clean CoNLL file: {out_path}")

csv_to_conll_clean(train_csv, os.path.join(out_dir, "train.conll"))
csv_to_conll_clean(test_csv, os.path.join(out_dir, "test.conll"))


✅ Wrote clean CoNLL file: /content/fv_prepared_cleaned/train.conll
✅ Wrote clean CoNLL file: /content/fv_prepared_cleaned/test.conll


**create validation split**

In [ ]:
from sklearn.model_selection import train_test_split

with open(os.path.join(out_dir, "train.conll"), encoding="utf-8") as f:
    sents = f.read().strip().split("\n\n")

train_sents, dev_sents = train_test_split(sents, test_size=0.2, random_state=42)

with open(os.path.join(out_dir, "train.conll"), "w", encoding="utf-8") as f:
    f.write("\n\n".join(train_sents) + "\n\n")
with open(os.path.join(out_dir, "dev.conll"), "w", encoding="utf-8") as f:
    f.write("\n\n".join(dev_sents) + "\n\n")

print("Recreated train/dev/test splits successfully.")


✅ Recreated train/dev/test splits successfully.


**Step 3 — Train the BiLSTM-CRF with Flair**

In [ ]:
from flair.datasets import ColumnCorpus
from flair.embeddings import WordEmbeddings, CharacterEmbeddings, StackedEmbeddings
from flair.models import SequenceTagger
from flair.trainers import ModelTrainer

columns = {0: "text", 1: "ner"}
corpus = ColumnCorpus(out_dir, columns,
                      train_file="train.conll",
                      dev_file="dev.conll",
                      test_file="test.conll")

word_embeddings = WordEmbeddings('glove')
char_embeddings = CharacterEmbeddings()
embeddings = StackedEmbeddings([word_embeddings, char_embeddings])

tag_dictionary = corpus.make_label_dictionary(label_type="ner")

tagger = SequenceTagger(
    hidden_size=256,
    embeddings=embeddings,
    tag_dictionary=tag_dictionary,
    tag_type="ner",
    use_crf=True
)

trainer = ModelTrainer(tagger, corpus)
trainer.train(
    base_path="/content/bilstm_crf_cleaned",
    learning_rate=0.1,
    mini_batch_size=32,
    max_epochs=25,
    embeddings_storage_mode='gpu'
)


2025-10-29 20:35:17,370 Reading data from /content/fv_prepared_cleaned
2025-10-29 20:35:17,376 Train: /content/fv_prepared_cleaned/train.conll
2025-10-29 20:35:17,377 Dev: /content/fv_prepared_cleaned/dev.conll
2025-10-29 20:35:17,379 Test: /content/fv_prepared_cleaned/test.conll
2025-10-29 20:35:39,458 Computing label dictionary. Progress:


0it [00:00, ?it/s]
13064it [00:00, 33840.99it/s]

2025-10-29 20:35:39,852 Dictionary created for label 'ner' with 21 values: vehicle_color (seen 15504 times), vehicle_brand (seen 14366 times), vehicle_model (seen 14366 times), vehicle_location (seen 7130 times), vehicle_orientation (seen 6616 times), vehicle_velocity (seen 6467 times), vehicle_type-sedan (seen 3112 times), vehicle_type-motorcycle (seen 2059 times), vehicle_type-suv (seen 2035 times), vehicle_type-sports_car (seen 1321 times), vehicle_type-hatchback (seen 1280 times), vehicle_type (seen 1212 times), vehicle_type-vintage_car (seen 1126 times), vehicle_type-coupe (seen 751 times), vehicle_type-truck (seen 682 times), vehicle_type-mpv (seen 489 times), vehicle_type-van (seen 448 times), vehicle_type-estate_car (seen 445 times), vehicle_type-bus (seen 367 times), vehicle_type-roadster (seen 303 times)
2025-10-29 20:35:39,853 SequenceTagger predicts: Dictionary with 85 tags: O, S-vehicle_color, B-vehicle_color, E-vehicle_color, I-vehicle_color, S-vehicle_brand, B-vehicle_br

2025-10-29 20:35:39,867 ----------------------------------------------------------------------------------------------------
2025-10-29 20:35:39,868 Model: "SequenceTagger(
  (embeddings): StackedEmbeddings(
    (list_embedding_0): WordEmbeddings(
      'glove'
      (embedding): Embedding(400001, 100)
    )
    (list_embedding_1): CharacterEmbeddings(
      (char_embedding): Embedding(275, 25)
      (char_rnn): LSTM(25, 25, bidirectional=True)
    )
  )
  (word_dropout): WordDropout(p=0.05)
  (locked_dropout): LockedDropout(p=0.5)
  (embedding2nn): Linear(in_features=150, out_features=150, bias=True)
  (rnn): LSTM(150, 256, batch_first=True, bidirectional=True)
  (linear): Linear(in_features=512, out_features=87, bias=True)
  (loss_function): ViterbiLoss()
  (crf): CRF()
)"
2025-10-29 20:35:39,869 ----------------------------------------------------------------------------------------------------
2025-10-29 20:35:39,870 Corpus: 13064 train + 3266 dev + 15917 test sentences
2025-10-29 

/usr/local/lib/python3.12/dist-packages/flair/trainers/trainer.py:499: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp and flair.device.type != "cpu")


...

100%|██████████| 52/52 [00:24<00:00,  2.14it/s]


2025-10-29 22:06:04,895 DEV : loss 0.08617530018091202 - f1-score (micro avg)  0.7525
2025-10-29 22:06:04,971  - 2 epochs without improvement
2025-10-29 22:06:04,972 ----------------------------------------------------------------------------------------------------
2025-10-29 22:06:40,448 epoch 15 - iter 40/409 - loss 0.09158917 - time (sec): 35.47 - samples/sec: 1516.75 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:07:16,162 epoch 15 - iter 80/409 - loss 0.09095089 - time (sec): 71.19 - samples/sec: 1512.70 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:07:51,409 epoch 15 - iter 120/409 - loss 0.08930901 - time (sec): 106.44 - samples/sec: 1521.06 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:08:26,337 epoch 15 - iter 160/409 - loss 0.09050340 - time (sec): 141.36 - samples/sec: 1520.17 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:09:02,450 epoch 15 - iter 200/409 - loss 0.08979249 - time (sec): 177.48 - samples/sec: 1520.41 - lr: 0.050000 - momentum: 0.000000
2025-10

100%|██████████| 52/52 [00:23<00:00,  2.18it/s]

2025-10-29 22:12:28,404 DEV : loss 0.08017941564321518 - f1-score (micro avg)  0.7239


2025-10-29 22:12:28,494  - 3 epochs without improvement
2025-10-29 22:12:28,495 ----------------------------------------------------------------------------------------------------
2025-10-29 22:13:03,514 epoch 16 - iter 40/409 - loss 0.08748065 - time (sec): 35.02 - samples/sec: 1520.11 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:13:37,042 epoch 16 - iter 80/409 - loss 0.08888434 - time (sec): 68.55 - samples/sec: 1549.70 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:14:12,134 epoch 16 - iter 120/409 - loss 0.08782896 - time (sec): 103.64 - samples/sec: 1544.20 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:14:47,698 epoch 16 - iter 160/409 - loss 0.08774110 - time (sec): 139.20 - samples/sec: 1543.81 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:15:22,509 epoch 16 - iter 200/409 - loss 0.08814014 - time (sec): 174.01 - samples/sec: 1546.41 - lr: 0.050000 - momentum: 0.000000
2025-10-29 22:15:57,771 epoch 16 - iter 240/409 - loss 0.08841634 - time (sec): 209.27 - samp

In [ ]:
from google.colab import files

# Path to your model directory
model_dir = "/content/bilstm_crf_cleaned"

# Download key files one by one
files.download(f"{model_dir}/best-model.pt")
files.download(f"{model_dir}/final-model.pt")
files.download(f"{model_dir}/training.log")
files.download(f"{model_dir}/loss.tsv")
files.download(f"{model_dir}/dev.tsv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import torch

# Temporary patch for PyTorch 2.6 to make Flair compatible
torch_load_original = torch.load

def torch_load_compat(*args, **kwargs):
    kwargs["weights_only"] = False
    return torch_load_original(*args, **kwargs)

torch.load = torch_load_compat
print(" Patched torch.load to allow Flair model loading.")


✅ Patched torch.load to allow Flair model loading.


In [ ]:
from flair.models import SequenceTagger
from flair.datasets import ColumnCorpus

model_path = "/content/bilstm_crf_cleaned/best-model.pt"
tagger = SequenceTagger.load(model_path)
print(" Model loaded successfully!")

# Reload dataset
columns = {0: "text", 1: "ner"}
corpus = ColumnCorpus(
    "/content/fv_prepared_cleaned",
    columns,
    train_file="train.conll",
    dev_file="dev.conll",
    test_file="test.conll"
)


2025-10-29 22:21:21,289 SequenceTagger predicts: Dictionary with 87 tags: O, S-vehicle_color, B-vehicle_color, E-vehicle_color, I-vehicle_color, S-vehicle_brand, B-vehicle_brand, E-vehicle_brand, I-vehicle_brand, S-vehicle_model, B-vehicle_model, E-vehicle_model, I-vehicle_model, S-vehicle_location, B-vehicle_location, E-vehicle_location, I-vehicle_location, S-vehicle_orientation, B-vehicle_orientation, E-vehicle_orientation, I-vehicle_orientation, S-vehicle_velocity, B-vehicle_velocity, E-vehicle_velocity, I-vehicle_velocity, S-vehicle_type-sedan, B-vehicle_type-sedan, E-vehicle_type-sedan, I-vehicle_type-sedan, S-vehicle_type-motorcycle, B-vehicle_type-motorcycle, E-vehicle_type-motorcycle, I-vehicle_type-motorcycle, S-vehicle_type-suv, B-vehicle_type-suv, E-vehicle_type-suv, I-vehicle_type-suv, S-vehicle_type-sports_car, B-vehicle_type-sports_car, E-vehicle_type-sports_car, I-vehicle_type-sports_car, S-vehicle_type-hatchback, B-vehicle_type-hatchback, E-vehicle_type-hatchback, I-veh

**Evaluation & Inference on Test Set**

In [ ]:
# Evaluate the model on the test set
result = tagger.evaluate(corpus.test, gold_label_type="ner")
print(result.detailed_results)


100%|██████████| 498/498 [02:56<00:00,  2.81it/s]



Results:
- F-score (micro) 0.4438
- F-score (macro) 0.4577
- Accuracy 0.2908

By class:
                          precision    recall  f1-score   support

           vehicle_color     0.0540    0.0728    0.0620     18809
           vehicle_model     0.3942    0.8835    0.5452     10721
           vehicle_brand     0.0220    0.5616    0.0424       942
        vehicle_location     0.6893    0.9975    0.8153      8501
     vehicle_orientation     0.6881    0.9871    0.8109      8187
        vehicle_velocity     0.6802    0.9952    0.8081      7979
      vehicle_type-sedan     0.5449    0.3908    0.4552      3664
 vehicle_type-motorcycle     0.6696    0.6998    0.6844      2432
        vehicle_type-suv     0.5111    0.4527    0.4801      2536
 vehicle_type-sports_car     0.7967    0.4579    0.5815      1566
vehicle_type-vintage_car     0.6884    0.3570    0.4702      1392
            vehicle_type     0.4045    0.9948    0.5752       573
  vehicle_type-hatchback     0.5152    0.1025    0.1

In [ ]:
!ls -lh /content/bilstm_crf_cleaned


total 327M
-rw-r--r-- 1 root root 162M Oct 29 21:53 best-model.pt
-rw-r--r-- 1 root root 2.4M Oct 29 22:12 dev.tsv
-rw-r--r-- 1 root root 162M Oct 29 22:16 final-model.pt
-rw-r--r-- 1 root root 1001 Oct 29 22:12 loss.tsv
-rw-r--r-- 1 root root  33K Oct 29 22:16 training.log
